<a href="https://colab.research.google.com/github/hjiwoong/DL/blob/main/day22_practice4_%EC%8B%A4%EB%AC%B4%ED%98%95Agent_Function_Calling_SDK_tools.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# RAG를 '도구 중 하나'로 품고, 도구를 호출하는 Agent를 LangChain 도구(@tool)와 OpenAI 호환 Function Calling(tools / tool_calls)으로 만든다

In [2]:
!pip install -q langchain langchain-core langchain-openai langchain-chroma langchain-text-splitters langchain-google-genai openai pypdf

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 kB 1.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 81.6/81.6 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 395.5/395.5 kB 13.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 60.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 25.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 10.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 264.6/264.6 kB 11.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.5/7.5 MB 61.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.6/23.6 MB 27.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.1

In [18]:
import os, requests
import torch
import torch.nn.functional as F
import json

def _get_api_key():
  try:
    from google.colab import userdata
    return userdata.get("VISION_API_KEY")
  except Exception:
    return os.environ.get("VISION_API_KEY", "") # Colab이 아니거나 Secret이 없을 때 환경변수 딕셔너리에서 꺼냄

API_KEY = _get_api_key()
# API 호출 - openAI 호환 api 형식으로 코드 작성
GEMINI_BASE_URL = "https://generativelanguage.googleapis.com/v1beta/openai" # Gemini OpenAI 호환

In [4]:
# 신고DB
REPORTS_DB = [
    {"id": 12, "hazard_type": "포트홀", "severity": 4, "location": "강남대로 사거리", "status": "처리중"},
    {"id": 15, "hazard_type": "포트홀", "severity": 4, "location": "서초대로 12",     "status": "접수"},
    {"id": 18, "hazard_type": "포트홀", "severity": 3, "location": "충무로 30",     "status": "완료"},
    {"id": 21, "hazard_type": "균열",   "severity": 2, "location": "을지로 골목",   "status": "접수"},
    {"id": 25, "hazard_type": "맨홀",   "severity": 3, "location": "안국역 3번출구", "status": "처리중"},
]

In [5]:
# 셀 1. 문서 로드 : PDF는 pypdf로 페이지별 텍스트 추출
import glob, os
from langchain_core.documents import Document # langchain은 Document 객체를 입력으로 받기 때문

def load_documents(folder = "/content/drive/MyDrive/KWU/DL/knowledge"):
  docs = []
  for path in sorted(glob.glob(os.path.join(folder, "*"))): # path: 파일 하나의 전체 경로
    name = os.path.basename(path) # 파일명
    if path.endswith(".pdf"):
      from pypdf import PdfReader # 함수/조건문 안에서도 import 가능
      reader = PdfReader(path) # pages 속성에 페이지들이 들어있음
      for i, page in enumerate(reader.pages): # 페이지 번호, 해당 페이지 한 장
        text = page.extract_text() # 글자만 문자열로, 스캔 이미지 PDF면 "" 빈 문자열로
        if text and text.strip(): # A and B 일 때 A가 false면 B는 안봄
          docs.append(Document(page_content=text,metadata={"source":name, "page": i}))
    elif path.endswith((".text", ".md")):
      with open(path, encoding="utf-8") as f:
        text = f.read().strip()
      if text:
        docs.append(Document(page_content=text, metadata={"source":name}))
  return docs

documents = load_documents()
print(f"1. 로드: 문서 {len(documents)}개") # Document 객체 갯수
for d in documents:
  print(f" [{d.metadata["source"]}] {len(d.page_content)}자")

1. 로드: 문서 5개
 [01_도로법_포트홀.pdf] 108자
 [02_맨홀_지침.pdf] 83자
 [03_균열_기준.pdf] 85자
 [04_신고_절차.pdf] 73자
 [05_종합_도로관리_규정.pdf] 632자


In [6]:
# 셀 2. 청킹 : 긴 문서를 검색 단위로 자른다
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=100)
chunks = splitter.split_documents(documents)
print(f"2. 청킹: 문서 {len(documents)}개 → 청크 {len(chunks)}개")

2. 청킹: 문서 5개 → 청크 6개


In [7]:
# 셀 3. 임베딩 + 벡터DB - Gemini 임베딩을 Chroma 벡터 DB에
from langchain_openai import OpenAIEmbeddings
from langchain_chroma import Chroma

# 임베딩 설정 (Gemini)
# OpenAIEmbeddings에 base_url만 Gemini로 바꾸면 Gemini 임베딩이 쓰인다 (OpenAI 호환)
embeddings = OpenAIEmbeddings(
    model = "gemini-embedding-2",
    base_url = GEMINI_BASE_URL,
    api_key = API_KEY,
    check_embedding_ctx_length = False, # OpenAI 전용 토큰검사 끔
)

# 벡터DB 구축 (Chroma)
vectorstore = Chroma.from_documents( # 청크들을 임베딩해서 벡터DB에 적재
    documents = chunks,
    embedding = embeddings,
    collection_name = "road_regulations",
    persist_directory = "/content/drive/MyDrive/KWU/DL/chroma_db", # 디스크에 영속 저장
)
print(f"3. 벡터DB 구축 완료: 청크 {len(chunks)}개 적재 (Gemini 임베딩)")

# retriever - 벡터DB를 '검색기' 객체로, k=3: 상위 3개 청크
retriever = vectorstore.as_retriever(search_kwargs = {"k":3})

3. 벡터DB 구축 완료: 청크 6개 적재 (Gemini 임베딩)


In [14]:
# 실습 2에서 도구 3개를 파이썬 함수로 만들고 설명서는 프롬프트 글로 따로 작성했는데
# @tool데코레이터는 이 둘을 합친 것 : 함수이름 + 타입힌트 + docstring이 곧 도구 설명서(스키마)

from langchain_core.tools import tool
def format_docs(docs):
  return "\n\n".join(d.page_content for d in docs) # 한 덩어리로

# 도구 1 : 규정 검색
@tool(parse_docstring=True, response_format="content_and_artifact") # parse_docstring=True는 docstring의  Args 설명을 인자 설명으로 파싱
def search_regulation(query: str): # response_format = (LLM에게 줄 글, 우리가 챙길 데이터)
  """ 도로 위험 처리 규정 문서를 검색한다. 처리 기한, 절차, 담당 부서 등 '규정'을 묻는 질문에 사용한다. 신고 건수나 특정 신고의 상태에는 사용하지 않는다.

  Args:
      query: 검색할 질문 또는 키워드 (예: '심각도 4 포트홀 처리 기한')
  """
  docs = retriever.invoke(query) # top3: list[Document]
  sources = sorted({d.metadata.get("source", "?") for d in docs})
  return format_docs(docs), sources

# 도구 2 : 신고 조회
@tool(parse_docstring=True)
def query_reports(hazard_type: str, min_severity: int) -> dict:
  """ 접수된 신고의 건수와 목록을 조회한다. '몇 건', '목록' 같은 현황 질문에 사용한다.

  Args:
      hazard_type: 위험 종류 (포트홀, 균열, 맨홀 중 하나). 종류 조건이 없으면 '전체'
      min_severity: 최소 심각도 (1-5 점수). 심각도 조건이 없으면 1
  """

  rows = [r for r in REPORTS_DB if (hazard_type in ("", "전체") or r["hazard_type"] == hazard_type) and r["severity"] >= int(min_severity)]
  return {"count": len(rows), "items": [f"#{r['id']} {r['location']} (심각도 {r['severity']}, {r['status']})" for r in rows]}

# 도구 3 : 상태 조회
@tool(parse_docstring=True)
def get_report_status(report_id: int) -> str:
  """ 신고번호로 특정 신고 1건의 처리 상태(접수/처리중/완료)를 조회한다.

  Args:
      report_id: 신고번호 (예: 12)
  """
  for r in REPORTS_DB:
    if r["id"] == int(report_id):
      return f"#{r['id']} {r['hazard_type']} ({r['location']} - 현재{r['status']})"
  return f"신고번호 {report_id} 를 찾을 수 없습니다."

tools = [search_regulation, query_reports, get_report_status]
tools_by_name = {t.name: t for t in tools} # (이름: 도구)
print("도구 3개 등록:")
for t in tools:
  print(f" - {t.name:18s} 인자 {list(t.args.keys())}")
print("\n단독 테스트:", get_report_status.invoke({"report_id": 12}))

도구 3개 등록:
 - search_regulation  인자 ['query']
 - query_reports      인자 ['hazard_type', 'min_severity']
 - get_report_status  인자 ['report_id']

단독 테스트: #12 포트홀 (강남대로 사거리 - 현재처리중)


In [19]:
# 셀 5. 스키마 확인 : @tool이 만들어 준것 (API가 요구하는 JSON Schema 로 바꿔준다. 실제 LLM에 전달되는 tools 파라미터 모양)
from langchain_core.utils.function_calling import convert_to_openai_tool
schema = convert_to_openai_tool(query_reports)
print(json.dumps(schema, ensure_ascii=False, indent=2))

{
  "type": "function",
  "function": {
    "name": "query_reports",
    "description": "접수된 신고의 건수와 목록을 조회한다. '몇 건', '목록' 같은 현황 질문에 사용한다.",
    "parameters": {
      "properties": {
        "hazard_type": {
          "description": "위험 종류 (포트홀, 균열, 맨홀 중 하나). 종류 조건이 없으면 '전체'",
          "type": "string"
        },
        "min_severity": {
          "description": "최소 심각도 (1-5 점수). 심각도 조건이 없으면 1",
          "type": "integer"
        }
      },
      "required": [
        "hazard_type",
        "min_severity"
      ],
      "type": "object"
    }
  }
}


```
user 메시지 ─► LLM ─► message(tool_calls=[{id, function:{name, arguments}}])
                                  │  우리 코드가 실행
                                  ▼
          {"role":"tool", "tool_call_id": id, "content": 결과} ─► LLM ─► message(content="최종 답")
```

In [21]:
# 셀 6. 스키마들을 모아 tools를 보낸다. 응답은 글이 아니라 message 객체
from openai import OpenAI
AGENT_MODEL = "gemini-3.5-flash-lite"
client = OpenAI(api_key=API_KEY, base_url=GEMINI_BASE_URL, max_retries=3) # 재시도3회
TOOLS_SCHEMA = [convert_to_openai_tool(t) for t in tools] # 도구 3개의 스키마 리스트
SYSTEM = """너는 도로 신고 시스템 상담 Agent다.
- 신고 건수, 목록은 query_reports, 특정 신고의 상태는 get_report_status, 처리 규정, 기한, 절차는 search_regulation을 사용한다
- 규정 질문은 search_regulation 결과에 근거해서만 답하고, 결과에 없으면 "규정에 없습니다"라고 답한다.
- 도구가 필요 없는 인사, 잡담은 도구 없이 바로 답한다. 답은 한구겅로 두 문장 이내."""
def call_llm(messages): # LLM 호출 함수
  resp = client.chat.completions.create(
      model = AGENT_MODEL, messages=messages,
      tools = TOOLS_SCHEMA, tool_choice = "auto", # auto = 도구를 쓸지 말지 LLM이 스스로 결정
      temperature = 0.2
  )
  return resp.choices[0].message # 원본 message

messages = [{"role": "system", "content":SYSTEM},
            {"role": "user", "content": "포트홀 신고가 몇 건이에요"}]

msg = call_llm(messages) # LLM의 1차 응답
print("1. tool_calls:", [(tc.function.name, tc.function.arguments) for tc in msg.tool_calls])

tc = msg.tool_calls[0] # 첫 번째 호출만 처리
args = json.loads(tc.function.arguments) # arguments는 JSON '문자열' → dict로
result = tools_by_name[tc.function.name].invoke(args) # 우리 코드가 도구 실행
print(f"2. 실행: {tc.function.name}({args}) → {result}")

messages.append(msg)
messages.append({"role": "tool", "tool_call_id": tc.id, "name": tc.function.name, "content": json.dumps(result, ensure_ascii=False)})

final = call_llm(messages)
print("3. 최종 답변:", final.content)

1. tool_calls: [('query_reports', '{"min_severity":"1","hazard_type":"포트홀"}')]
2. 실행: query_reports({'min_severity': '1', 'hazard_type': '포트홀'}) → {'count': 3, 'items': ['#12 강남대로 사거리 (심각도 4, 처리중)', '#15 서초대로 12 (심각도 4, 접수)', '#18 충무로 30 (심각도 3, 완료)']}
3. 최종 답변: 현재 포트홀 신고는 총 3건이 접수되어 있습니다. 강남대로, 서초대로, 충무로 인근의 신고가 포함되어 있습니다.


In [22]:
# 셀 7. Agent 루프
# 병렬 호출 - 한 턴에 tool_calls가 여러 개 올 수 있다
# 오류를 LLM에게 돌려주기 - LLM이 인자를 고쳐 재시도
# 안전장치 - 최대 턴 수

MAX_TURNS = 5 # 안전 장치1: 무한루프 방지

def _text(content): # LLM 응답을 문자열로
  if isinstance(content, list):
    return "".join(p.get("text", "") if isinstance(p, dict) else str(p) for p in content)
  return content or "" # None이면 ""

def execute_tool_call(tc): # LLM이 고른 도구를 실제로 실행
  name = tc.function.name
  if name not in tools_by_name: # 안전 장치 2: 목록이 없는 도구는 실행 금지
    return f"오류: 알 수 없는 도구 '{name}', 사용가능: {list(tools_by_name)}", None
  try:
    args = json.loads(tc.function.arguments or "{}") # {dict}로
    if not isinstance(args, dict):
      raise ValueError("arguments는 JSON 객체여야 합니다")
    tm = tools_by_name[name].invoke({"name":name, "args":args, "id":tc.id, "type":"tool_call"}) # tool_call로 invoke하면 ToolMessage
    return str(tm.content), getattr(tm, "artifact", None)
  except Exception as e:
    return f"도구 실행 오류({type(e).__name__}): {str(e)[:200]}", None

def run_agent(question, verbose=True): # Agent 전체 루프
  messages = [{"role": "system", "content":SYSTEM},
              {"role": "user", "content": question}]
  used_tools, sources = [], []
  for turn in range(MAX_TURNS):
    msg = call_llm(messages)
    calls = msg.tool_calls or [] # 이번 턴의 도구 호출
    if not calls:
      return {"answer": _text(msg.content), "used_tools": used_tools, # 마지막 답과 함께 어떤 도구를 썼는지, 어떤 출처를 참고했는지, 몇 턴이 걸쳤는지
              "sources": sorted(set(sources)), "turns": turn + 1}
    messages.append(msg) # 응답 message '원본 그대로' 먼저 기록
    for tc in calls: # 한 턴에 여러 개 도구 호출
      out, artifact = execute_tool_call(tc) # 도구 실행 결과
      used_tools.append(tc.function.name)
      if isinstance(artifact, list):
        sources.extend(artifact) # sources에 이어붙임
      if verbose: print(f"[턴{turn+1}] {tc.function.name}({tc.function.arguments}) → {out[:56]}") # 중간 과정을 보여주는
      messages.append({"role":"tool", "tool_call_id":tc.id, "name":tc.function.name, "content":out})
      # 도구 결과는 role을 "tool"로 넣고 tool_call_id로 요청과 짝지어야 API가 정상 처리
  return {"answer": "여러 번 시도했지만 답을 완성하지 못했습니다.",
          "used_tools": used_tools, "sources": sorted(set(sources)), "turns": MAX_TURNS}

In [23]:
# 셀 8. 실행 - 질문 유형별로 다른 도구를 집는가
질문들 = [
    "포트홀 신고 몇 건이나 있어요?",
    "심각도 4 포트홀은 언제 처리되나요?",
    "12번 신고 상태 어떻게 돼요?",
    "심각도 4 포트홀이 몇 건이고, 처리 기한 규정도 알려줘",
    "신고하면 보상금은 얼마인가요?",
    "안녕하세요",
]
for q in 질문들:
  print(f"질문: {q}")
  out = run_agent(q)
  print(f"답변: {out['answer']}")
  print(f"사용 도구: {out['used_tools'] or '(없음)'} / 턴 수: {out['turns']}")

질문: 포트홀 신고 몇 건이나 있어요?
[턴1] query_reports({"hazard_type":"포트홀","min_severity":1}) → {"count": 3, "items": ["#12 강남대로 사거리 (심각도 4, 처리중)", "#15
답변: 현재 접수된 포트홀 신고는 총 3건입니다. 자세한 목록과 처리 상태는 시스템에서 확인하실 수 있습니다.
사용 도구: ['query_reports'] / 턴 수: 2
질문: 심각도 4 포트홀은 언제 처리되나요?
[턴1] search_regulation({"query":"심각도 4 포트홀 처리 기한"}) → 01 도로법 포트홀
도로법 시행령 제55조에 따라 도로 관리청은 포트홀 등 도로 파손 발견 시 24시
답변: 규정에 따라 포트홀은 발견 시 24시간 이내 응급 복구 조치를 시행해야 하며, 심각도 4 이상은 긴급 보수팀이 우선 배정됩니다.
사용 도구: ['search_regulation'] / 턴 수: 2
질문: 12번 신고 상태 어떻게 돼요?
[턴1] get_report_status({"report_id":12}) → #12 포트홀 (강남대로 사거리 - 현재처리중)
답변: 12번 신고는 현재 '처리중' 상태입니다. 조금만 기다려 주시면 신속하게 조치하겠습니다.
사용 도구: ['get_report_status'] / 턴 수: 2
질문: 심각도 4 포트홀이 몇 건이고, 처리 기한 규정도 알려줘
[턴1] query_reports({"hazard_type":"포트홀","min_severity":4}) → {"count": 2, "items": ["#12 강남대로 사거리 (심각도 4, 처리중)", "#15
[턴1] search_regulation({"query":"포트홀 심각도 4 처리 기한"}) → 01 도로법 포트홀
도로법 시행령 제55조에 따라 도로 관리청은 포트홀 등 도로 파손 발견 시 24시
답변: 현재 심각도 4인 포트홀은 총 2건 접수되어 있습니다. 규정에 따라 포트홀 발견 시 24시간 이내 응급 복구 

In [24]:
# 셀 9. create_agent (랭체인 프레임워크가 루프를 대신)
# 루프(도구 호출 → 도구 실행 → LLM재질의 → 종료 판단 + 안전장치)를 LangChain 함수 대신 한 줄로
try:
  from langchain.agents import create_agent
  from langchain_google_genai import ChatGoogleGenerativeAI # 제미나이 전용
  native_llm = ChatGoogleGenerativeAI(model=AGENT_MODEL, google_api_key=API_KEY, temperature=0.2)
  agent = create_agent(model = native_llm, tools=tools, system_prompt=SYSTEM) # 루프가 내장된 Agent
  out = agent.invoke({"messages": [{"role":"user", "content":"맨홀 신고하면 어디서 처리해요?"}]})
  msgs = out["messages"]
  used = [tc["name"] for m in msgs if getattr(m, "tool_calls", None) for tc in m.tool_calls]
  print("답변:", _text(msgs[-1].content))
  print("사용 도구:", used or "(없음)", f"/ 메시지 {len(msgs)}개")
except Exception as e:
  print(f"create_agent 실행 실패({type(e).__name__}: {str(e)[:150]})")

/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3908: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(
/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3908: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


답변: 관할 지자체에서 처리하도록 규정되어 있습니다. 추가로 궁금한 점이 있으신가요?
사용 도구: ['search_regulation'] / 메시지 4개


In [25]:
print(out)

{'messages': [HumanMessage(content='맨홀 신고하면 어디서 처리해요?', additional_kwargs={}, response_metadata={}, id='db81d768-2606-411e-9495-cc92c3040eae'), AIMessage(content=[], additional_kwargs={'function_call': {'name': 'search_regulation', 'arguments': '{"query": "\\ub9e8\\ud640 \\ucc98\\ub9ac \\ub2f4\\ub2f9 \\ubd80\\uc11c \\uaddc\\uc815"}'}, '__gemini_function_call_thought_signatures__': {'call_257206': 'EmAKXgFpFH0TV41EsUJH6U4vtxT5RbWwWK8bL38iUpzocgYke+Jr2lpMmYeWD/RSA0ip4OKSSPkoSE8gD97h2TRSQzApZSv60XWnK8AIZmpwJiBm9IFG9BNeXmcK9Zg/cS8='}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.5-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--01a115c1-05dc-78e2-8dc7-848c6b2aa7de-0', tool_calls=[{'name': 'search_regulation', 'args': {'query': '맨홀 처리 담당 부서 규정'}, 'id': 'call_257206', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 458, 'output_tokens': 23, 'total_tokens': 481, 'input_token_details': {'cache_read': 0}}), 

In [26]:
# 셀 10. 출처까지 함께 (실무 필수)
result = run_agent("심각도 4 포트홀은 언제 처리되나요?", verbose = False)
print("답변:", result["answer"])
print("사용 도구:", result["used_tools"])
print("출처:", result["sources"])

답변: 심각도 4 이상의 포트홀은 도로법 시행령에 따라 발견 시 24시간 이내 응급 복구 조치를 시행하며, 긴급 보수팀이 우선 배정됩니다.
사용 도구: ['search_regulation']
출처: ['01_도로법_포트홀.pdf']
